In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

device = torch.device("cpu")
torch.set_num_threads(4)  # i5 de 11ª geração: 4 núcleos físicos (teste 4 e 8 depois)

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

PyTorch: 2.15.0+cpu
Dispositivo: cpu


In [2]:
with open("../dados/dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Caracteres muito raros (emojis, símbolos soltos) só aumentam o vocabulário
contagem = Counter(text)
text = "".join(c if contagem[c] >= 5 else "?" for c in text)

print("Número de caracteres:", len(text))
print()
print(text[:500])

Número de caracteres: 2693666

@@@ ARQUIVO: spring-data-examples/jpa/showcase/src/test/java/example/springdata/jpa/showcase/after/AccountRepositoryIntegrationTest.java @@@
package example.springdata.jpa.showcase.after;

import static org.assertj.core.api.Assertions.*;

import example.springdata.jpa.showcase.core.Account;

import java.util.Collections;

import org.junit.jupiter.api.Test;

import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.boot.test.context.SpringBootTest;
import org.sprin


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

print("|V| =", vocab_size)
print([repr(c) for c in chars])

|V| = 97
["'\\n'", "' '", "'!'", '\'"\'', "'#'", "'$'", "'%'", "'&'", '"\'"', "'('", "')'", "'*'", "'+'", "','", "'-'", "'.'", "'/'", "'0'", "'1'", "'2'", "'3'", "'4'", "'5'", "'6'", "'7'", "'8'", "'9'", "':'", "';'", "'<'", "'='", "'>'", "'?'", "'@'", "'A'", "'B'", "'C'", "'D'", "'E'", "'F'", "'G'", "'H'", "'I'", "'J'", "'K'", "'L'", "'M'", "'N'", "'O'", "'P'", "'Q'", "'R'", "'S'", "'T'", "'U'", "'V'", "'W'", "'X'", "'Y'", "'Z'", "'['", "'\\\\'", "']'", "'_'", "'`'", "'a'", "'b'", "'c'", "'d'", "'e'", "'f'", "'g'", "'h'", "'i'", "'j'", "'k'", "'l'", "'m'", "'n'", "'o'", "'p'", "'q'", "'r'", "'s'", "'t'", "'u'", "'v'", "'w'", "'x'", "'y'", "'z'", "'{'", "'|'", "'}'", "'~'", "'©'", "'ü'"]


In [4]:
exemplo = '@RestController\npublic class Foo {\n    <div class="flex">'
codificado = encode(exemplo)
print("Tokens:", codificado[:20], "...")
print("Reconstrução igual ao original?", decode(codificado) == exemplo)

Tokens: [33, 51, 69, 83, 84, 36, 79, 78, 84, 82, 79, 76, 76, 69, 82, 0, 80, 85, 66, 76] ...
Reconstrução igual ao original? True


In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
print("Shape:", data.shape)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("Tokens de treino:", len(train_data))
print("Tokens de validação:", len(val_data))

Shape: torch.Size([2693666])
Tokens de treino: 2424299
Tokens de validação: 269367


In [6]:
import re

cabecalhos = re.findall(r"@@@ ARQUIVO: (.+?) @@@", text)
grupos = Counter("/".join(c.split("/")[:3]) for c in cabecalhos)

for grupo, qtd in grupos.most_common(8):
    print(f"{qtd:5d}  {grupo}")

   59  spring-data-examples/jpa/example
   49  spring-petclinic/src/main
   45  spring-data-examples/jdbc/howto
   45  spring-data-examples/cassandra/example
   38  spring-data-examples/mongodb/example
   24  flowbite/layouts/_shortcodes
   22  spring-data-examples/jpa/showcase
   21  spring-data-examples/jpa/multitenant


In [7]:
import math, time, csv, os

torch.manual_seed(1337)

d = 128           # dimensão do embedding
T = 128           # janela de contexto
L = 4             # blocos Transformer
h = 4             # cabeças de atenção
batch_size = 32

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(0, len(source) - T - 1, (batch_size,))
    X = torch.stack([source[i:i+T] for i in starts])
    Y = torch.stack([source[i+1:i+T+1] for i in starts])
    return X, Y

X, Y = get_batch("train")
print("X:", X.shape, "| Y:", Y.shape)   # esperado: (32, 128)

X: torch.Size([32, 128]) | Y: torch.Size([32, 128])


In [8]:
class Head(nn.Module):
    def __init__(self, d, head_dim, T):
        super().__init__()
        self.key = nn.Linear(d, head_dim, bias=False)
        self.query = nn.Linear(d, head_dim, bias=False)
        self.value = nn.Linear(d, head_dim, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))
        self.head_dim = head_dim

    def forward(self, x):                                   # x: (B, T, d)
        Q, K, V = self.query(x), self.key(x), self.value(x)  # cada um: (B, T, head_dim)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)  # (B, T, T)
        Tc = x.size(1)
        scores = scores.masked_fill(self.mask[:Tc, :Tc] == 0, float("-inf"))
        A = F.softmax(scores, dim=-1)                       # (B, T, T)
        return A @ V                                        # (B, T, head_dim)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        assert d % h == 0
        self.heads = nn.ModuleList([Head(d, d // h, T) for _ in range(h)])
        self.projection = nn.Linear(d, d)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)  # (B, T, d)
        return self.projection(out)


class FeedForward(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attention = MultiHeadAttention(d, h, T)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = FeedForward(d)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d, T, L, h):
        super().__init__()
        self.T = T
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, h, T) for _ in range(L)])
        self.ln_final = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab_size)

    def forward(self, idx):                                  # idx: (B, T)
        B, Tc = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(Tc, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_final(x))                # (B, T, vocab_size)

In [9]:
model = MiniGPT(vocab_size, d, T, L, h).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")

X, Y = get_batch("train")
logits = model(X)
loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
print("Loss inicial:", round(loss.item(), 3), "| esperado ≈", round(math.log(vocab_size), 3))

Parâmetros: 833,121
Loss inicial: 4.727 | esperado ≈ 4.575


In [10]:
t0 = time.time()
for _ in range(10):
    X, Y = get_batch("train")
    loss = loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
seg_por_passo = (time.time() - t0) / 10
print(f"{seg_por_passo:.2f} s/passo | 3000 passos ≈ {seg_por_passo * 3000 / 60:.0f} min")

0.26 s/passo | 3000 passos ≈ 13 min


In [11]:
import math, time, csv, os

torch.manual_seed(1337)

d = 128           # dimensão do embedding
T = 128           # janela de contexto
L = 4             # blocos Transformer
h = 4             # cabeças de atenção
batch_size = 32

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(0, len(source) - T - 1, (batch_size,))
    X = torch.stack([source[i:i+T] for i in starts])
    Y = torch.stack([source[i+1:i+T+1] for i in starts])
    return X, Y

X, Y = get_batch("train")
print("X:", X.shape, "| Y:", Y.shape)

X: torch.Size([32, 128]) | Y: torch.Size([32, 128])


In [12]:
class Head(nn.Module):
    def __init__(self, d, head_dim, T):
        super().__init__()
        self.key = nn.Linear(d, head_dim, bias=False)
        self.query = nn.Linear(d, head_dim, bias=False)
        self.value = nn.Linear(d, head_dim, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))
        self.head_dim = head_dim

    def forward(self, x):                                   # x: (B, T, d)
        Q, K, V = self.query(x), self.key(x), self.value(x)  # cada um: (B, T, head_dim)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)  # (B, T, T)
        Tc = x.size(1)
        scores = scores.masked_fill(self.mask[:Tc, :Tc] == 0, float("-inf"))
        A = F.softmax(scores, dim=-1)                       # (B, T, T)
        return A @ V                                        # (B, T, head_dim)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        assert d % h == 0
        self.heads = nn.ModuleList([Head(d, d // h, T) for _ in range(h)])
        self.projection = nn.Linear(d, d)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)  # (B, T, d)
        return self.projection(out)


class FeedForward(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attention = MultiHeadAttention(d, h, T)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = FeedForward(d)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d, T, L, h):
        super().__init__()
        self.T = T
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, h, T) for _ in range(L)])
        self.ln_final = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab_size)

    def forward(self, idx):                                  # idx: (B, T)
        B, Tc = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(Tc, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_final(x))                # (B, T, vocab_size)

In [15]:
model = MiniGPT(vocab_size, d, T, L, h).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")

X, Y = get_batch("train")
logits = model(X)
print("logits:", logits.shape)   # esperado: (32, 128, vocab_size)
loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
print("Loss inicial:", round(loss.item(), 3), "| esperado ≈", round(math.log(vocab_size), 3))

Parâmetros: 833,121
logits: torch.Size([32, 128, 97])
Loss inicial: 4.715 | esperado ≈ 4.575


In [14]:
t0 = time.time()
for _ in range(10):
    X, Y = get_batch("train")
    loss = loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
seg_por_passo = (time.time() - t0) / 10
print(f"{seg_por_passo:.2f} s/passo | 3000 passos ≈ {seg_por_passo * 3000 / 60:.0f} min")

0.23 s/passo | 3000 passos ≈ 11 min


In [17]:
from torch.optim.lr_scheduler import CosineAnnealingLR

os.makedirs("../logs", exist_ok=True)
os.makedirs("../modelos", exist_ok=True)

max_steps = 8000
eval_every = 250
scheduler = CosineAnnealingLR(optimizer, T_max=max_steps)

@torch.no_grad()
def estimate_loss(n=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        vals = []
        for _ in range(n):
            X, Y = get_batch(split)
            vals.append(loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1)).item())
        out[split] = sum(vals) / n
    model.train()
    return out

best_val = float("inf")

with open("../logs/treino.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "train_loss", "val_loss", "lr", "d", "T", "L", "h", "batch_size"])
    t0 = time.time()
    for step in range(max_steps + 1):
        if step % eval_every == 0:
            r = estimate_loss()
            lr_atual = optimizer.param_groups[0]["lr"]
            writer.writerow([step, round(r["train"], 4), round(r["val"], 4),
                             round(lr_atual, 6), d, T, L, h, batch_size])
            f.flush()
            marca = ""
            if r["val"] < best_val:
                best_val = r["val"]
                torch.save({"model": model.state_dict(), "chars": chars,
                            "hparams": dict(d=d, T=T, L=L, h=h)}, "../modelos/minigpt.pt")
                marca = " *salvo*"
            print(f"step {step:4d} | train {r['train']:.3f} | val {r['val']:.3f} | "
                  f"{(time.time() - t0) / 60:.1f} min{marca}")
        X, Y = get_batch("train")
        loss = loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

print("Fim. Melhor val loss:", round(best_val, 3))

step    0 | train 4.703 | val 4.641 | 0.1 min *salvo*
step  250 | train 2.161 | val 1.908 | 1.2 min *salvo*
step  500 | train 1.493 | val 1.391 | 2.3 min *salvo*
step  750 | train 1.204 | val 1.112 | 3.3 min *salvo*
step 1000 | train 1.060 | val 0.961 | 4.4 min *salvo*
step 1250 | train 0.947 | val 0.873 | 5.4 min *salvo*
step 1500 | train 0.867 | val 0.814 | 6.5 min *salvo*
step 1750 | train 0.794 | val 0.776 | 7.5 min *salvo*
step 2000 | train 0.793 | val 0.762 | 8.5 min *salvo*
step 2250 | train 0.784 | val 0.708 | 9.6 min *salvo*
step 2500 | train 0.706 | val 0.666 | 10.6 min *salvo*
step 2750 | train 0.744 | val 0.640 | 11.7 min *salvo*
step 3000 | train 0.702 | val 0.655 | 12.7 min
step 3250 | train 0.655 | val 0.655 | 13.7 min
step 3500 | train 0.648 | val 0.632 | 14.8 min *salvo*
step 3750 | train 0.627 | val 0.608 | 15.9 min *salvo*
step 4000 | train 0.663 | val 0.597 | 16.9 min *salvo*
step 4250 | train 0.603 | val 0.582 | 18.0 min *salvo*
step 4500 | train 0.578 | val 0.572 

In [18]:
with open("../logs/treino.csv") as f:
    linhas = list(csv.reader(f))
dados = linhas[1:]
for r in dados[::4] + dados[-1:]:
    print(f"step {r[0]:>5} | train {r[1]} | val {r[2]}")

step     0 | train 4.703 | val 4.6407
step  1000 | train 1.06 | val 0.9613
step  2000 | train 0.7931 | val 0.7619
step  3000 | train 0.7016 | val 0.655
step  4000 | train 0.6626 | val 0.5967
step  5000 | train 0.5564 | val 0.5654
step  6000 | train 0.5126 | val 0.527
step  7000 | train 0.4917 | val 0.5352
step  8000 | train 0.5039 | val 0.5131
step  8000 | train 0.5039 | val 0.5131


In [19]:
ckpt = torch.load("../modelos/minigpt.pt")
model.load_state_dict(ckpt["model"])
model.eval()

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.8, top_k=20):
    for _ in range(max_new_tokens):
        logits = model(idx[:, -model.T:])[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

def completar(prompt, n=400, **kw):
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    return decode(generate(model, idx, n, **kw)[0].tolist())

for p in ["@RestController\npublic class", "@Entity\npublic class", '<div class="flex', "SELECT "]:
    print("=" * 60)
    print(completar(p))

@RestController
public class CustomerRepository extends CrudRepository<User, INOND, MongoOperations) {

}
@@@ FIM @@@

@@@ AR0 ARQUIVO: spring-petclinic/src/main/java/org/springframework/samples/petclinic/vet/VisitConstructorResources/ApiGraalApplication.java @@@
package example.springdata.jpa.hibernatemultitenant.password.bulk.MongoOperations;

import org.springframework.data.jpa.core.core.schema.CreatedDate;
import org.spr
@Entity
public class ActiveProcessService {

    public User(Application.class) {

        Association.insert(fields.addMinion.class);

        var address = address.remum});

        var blogPosts
                                                }
          }
              0.90034523134, -0.0065252074, -0.00048662, 0.0310530392, -0.01325045, 0.048624987, 0.00768054, -0.003363723, 0.022142725,
            -0.005415075
<div class="flex justify-center gap-1.5">
     <label class="block px-2 py-1 text-sm font-medium text-white transition-opacity duration-300 bg-dark ro

In [20]:
def taxa_copia(amostra, janela=40):
    trechos = [amostra[i:i+janela] for i in range(0, len(amostra) - janela, 10)]
    achados = sum(1 for t in trechos if t in text)
    return achados / max(1, len(trechos))

for p in ["@RestController\npublic class", "@Entity\npublic class", '<div class="flex']:
    g = completar(p, n=600)
    print(repr(p), "->", f"{taxa_copia(g):.0%} dos trechos de 40 caracteres existem no dataset")

'@RestController\npublic class' -> 2% dos trechos de 40 caracteres existem no dataset
'@Entity\npublic class' -> 9% dos trechos de 40 caracteres existem no dataset
'<div class="flex' -> 62% dos trechos de 40 caracteres existem no dataset
